# Lab 4 – Data Quality Assessment & Preprocessing
**Dataset:** Medical Insurance Cost (`insurance.csv`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('insurance.csv')
print(f'Shape: {df.shape}')
df.head()

---
## Task 1 – Identify Data Quality Issues

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print()

print('=== Missing Values ===')
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Count': missing, '%': pct}))
print()

print('=== Duplicate Rows ===')
print(f'  Duplicates: {df.duplicated().sum()}')
print()

print('=== Descriptive Statistics ===')
df.describe().round(2)

In [ ]:
# Visualise charges distribution — check skewness
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['charges'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Charges — Raw Distribution')
axes[0].set_xlabel('Charges (USD)')

axes[1].hist(np.log1p(df['charges']), bins=40, color='darkorange', edgecolor='white')
axes[1].set_title('Charges — Log-Transformed')
axes[1].set_xlabel('log(1 + Charges)')
plt.tight_layout()
plt.show()

print(f'Charges skewness: {df["charges"].skew():.3f}  (>1 = right-skewed)')

**Identified Data Quality Issues:**
1. `charges` is **strongly right-skewed** (skewness > 1) — driven by the smoker high-cost cluster.
2. `bmi` contains **outliers** at the extremes (very low / very high values).
3. `charges` contains **outliers** — smokers with high BMI push values above $50,000.
4. Categorical columns (`sex`, `smoker`, `region`) need **encoding** before modelling.
5. **No missing values** in this dataset, but a strategy is demonstrated below.

---
## Task 2 – Missing Value Handling Strategy

In [ ]:
# Inject synthetic missing values to demonstrate the strategy
import random
random.seed(42)
df_missing = df.copy()
idx_bmi    = random.sample(range(len(df_missing)), 40)
idx_age    = random.sample(range(len(df_missing)), 20)
df_missing.loc[idx_bmi, 'bmi'] = np.nan
df_missing.loc[idx_age, 'age'] = np.nan

print('Missing after injection:')
print(df_missing.isnull().sum())

In [ ]:
# Strategy: Median imputation for bmi and age
df_missing['bmi'].fillna(df_missing['bmi'].median(), inplace=True)
df_missing['age'].fillna(df_missing['age'].median(), inplace=True)

print('Missing after imputation:')
print(df_missing.isnull().sum())

**Why Median Imputation?**  
`bmi` is slightly right-skewed and contains outliers — the median is robust to both, whereas the mean would be pulled upward by extreme values. Similarly, `age` has a roughly uniform distribution; the median gives the central tendency without being influenced by any cluster of extreme entries.  
For a production model, **group-specific medians** (e.g., by region or smoker status) would give even better estimates.

---
## Task 3 – Detect and Handle Outliers (IQR Method)

In [ ]:
df_clean = df.copy()
numerical = ['bmi', 'charges']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, col in enumerate(numerical):
    sns.boxplot(y=df_clean[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(f'{col} — Before')
plt.suptitle('Before Outlier Removal', fontsize=12)
plt.tight_layout(); plt.show()

original_len = len(df_clean)
for col in numerical:
    Q1, Q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
    IQR    = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out  = ((df_clean[col] < lo) | (df_clean[col] > hi)).sum()
    print(f'{col}: Q1={Q1:.2f} Q3={Q3:.2f} IQR={IQR:.2f} | bounds=[{lo:.2f}, {hi:.2f}] | outliers={n_out}')
    df_clean = df_clean[(df_clean[col] >= lo) & (df_clean[col] <= hi)]

print(f'\nRows removed : {original_len - len(df_clean)}')
print(f'Remaining rows: {len(df_clean)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, col in enumerate(numerical):
    sns.boxplot(y=df_clean[col], ax=axes[i], color='lightgreen')
    axes[i].set_title(f'{col} — After')
plt.suptitle('After Outlier Removal', fontsize=12)
plt.tight_layout(); plt.show()

---
## Task 4 – Normalize Numerical Features

In [ ]:
num_features = ['age', 'bmi', 'children', 'charges']
X_num = df_clean[num_features].copy()

# Min-Max Scaling
mm  = MinMaxScaler()
df_minmax = pd.DataFrame(mm.fit_transform(X_num),
                          columns=[c+'_mm' for c in num_features])

# Z-score Standardisation
zs  = StandardScaler()
df_zscore = pd.DataFrame(zs.fit_transform(X_num),
                           columns=[c+'_zs' for c in num_features])

print('Min-Max (first 5 rows):')
print(df_minmax.head().round(4))
print()
print('Z-score (first 5 rows):')
print(df_zscore.head().round(4))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, col in enumerate(num_features):
    axes[0][i].hist(df_minmax[col+'_mm'], bins=25, color='steelblue', edgecolor='white')
    axes[0][i].set_title(f'{col}\nMin-Max', fontsize=9)
    axes[1][i].hist(df_zscore[col+'_zs'], bins=25, color='darkorange', edgecolor='white')
    axes[1][i].set_title(f'{col}\nZ-score', fontsize=9)
plt.suptitle('Comparison: Min-Max Scaling vs Z-score Standardisation', fontsize=13)
plt.tight_layout(); plt.show()

---
## Task 5 – Principal Component Analysis (PCA)

In [ ]:
# Encode categoricals for PCA
df_pca = df_clean.copy()
df_pca['smoker_enc'] = (df_pca['smoker'] == 'yes').astype(int)
df_pca['sex_enc']    = (df_pca['sex']    == 'male').astype(int)
df_pca = pd.get_dummies(df_pca, columns=['region'], drop_first=True)
df_pca.drop(columns=['sex','smoker'], inplace=True)

X_pca = StandardScaler().fit_transform(df_pca)

pca = PCA()
pca.fit_transform(X_pca)

ev  = pca.explained_variance_ratio_
cum = np.cumsum(ev)

print('Component | Variance % | Cumulative %')
for i,(v,c) in enumerate(zip(ev,cum)):
    print(f'  PC{i+1:>2}   |   {v*100:5.2f}%   |   {c*100:6.2f}%')

plt.figure(figsize=(8, 4))
plt.bar(range(1,len(ev)+1), ev*100, alpha=0.7, color='steelblue', label='Individual')
plt.plot(range(1,len(ev)+1), cum*100, marker='o', color='tomato', label='Cumulative')
plt.axhline(95, color='green', linestyle='--', label='95% threshold')
plt.xlabel('Principal Component'); plt.ylabel('Explained Variance (%)')
plt.title('PCA — Explained Variance per Component')
plt.legend(); plt.tight_layout(); plt.show()

**PCA Interpretation:**  
The first 2–3 principal components capture the majority of variance. **PC1** is dominated by `charges` and `smoker_enc`, reflecting that smoking status and cost move together as the primary source of variation in the dataset. **PC2** captures the `age` + `bmi` axis — older and heavier individuals form their own cost pattern independent of smoking. Later components (PC4+) contribute little individually, suggesting that most predictive information is concentrated in a low-dimensional subspace. For this dataset with only ~8 features after encoding, PCA is not critical for dimensionality reduction but confirms that the features are not redundant.